# Run inference with exported classifiers

This notebook trains the same example classifiers used in the trainer notebooks, saves them, reloads them, and uses the reloaded models to make predictions. It is a guided demonstration of the inference workflow, not a correctness test.

Each section uses the real AGN-versus-star-forming Cloudy grids under `data/` and its corresponding training configuration. Run a section's cells from top to bottom. The exported artifacts are written to named directories in the current working directory so you can inspect or reuse them after the notebook finishes.


## SimpleTrainer: save, load, and predict

This example uses the same silver-level CSV files and configuration as `simpletrainer_examples.ipynb`. The source files contain Cloudy AGN models (class 0) and POPSTAR models (class 1).


In [ ]:
from pathlib import Path

import torch
import yaml
from torch.utils.data import random_split

from GalaxySpectrumClassifier import SimpleTrainer, TabularDataset, load_skops


def find_project_root():
    """Find the repository whether the kernel starts in root or notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "configs").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root")


simple_root = find_project_root()
with (simple_root / "configs/binary_classsifier_simple_example.yaml").open() as stream:
    simple_config = yaml.safe_load(stream)

# Resolve the dataset directory so the notebook works from any kernel location.
simple_config["dataset"]["path"] = str(simple_root / "data/silver/default")
simple_dataset = TabularDataset.from_config(simple_config["dataset"])
# Keep a held-out portion of the real data for the prediction demonstration.
simple_train, simple_test = random_split(
    simple_dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

### Train and save the classifier

The classifier keeps the training settings from `simpletrainer_examples.ipynb`. Only the output directory changes, making the saved model easy to find after the demonstration.


In [ ]:
simple_output_path = Path("inference_example_simple_classifier")
simple_trainer_config = simple_config["trainer"].copy()
simple_trainer_config["output_path"] = str(simple_output_path)

simple_trainer = SimpleTrainer.from_config(simple_trainer_config)
simple_trainer.fit(simple_train)

# Save the fitted classifier so it can be used in a separate Python session.
simple_model_path = simple_output_path / "classifier.skops"
simple_trainer.export_model(simple_model_path)

### Load the saved classifier and predict

Load the exported classifier, then pass held-out feature rows to `predict`. The result is one predicted class per row.


In [ ]:
from GalaxySpectrumClassifier import to_xy

# Convert the held-out dataset into the feature array accepted by the classifier.
simple_features, simple_labels = to_xy(simple_test)
simple_model = load_skops(simple_model_path)

simple_predictions = simple_model.predict(simple_features)
# Display predicted classes and their probabilities for the first held-out rows.
simple_predictions[:10], simple_model.predict_proba(simple_features[:10])

## EpochTrainer: save, load, and predict

This section uses the existing gold-level train, validation, and test splits from `epochtrainer_examples.ipynb`. It trains with the same configuration as that notebook, exports the trained classifier in two supported formats, and then loads each export for prediction.


In [ ]:
from pathlib import Path

import torch
import yaml

from GalaxySpectrumClassifier import EpochTrainer, load_default, load_torch


def find_binary_project_root():
    """Find the checked-out project and its real example data."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/gold/epoch_trainer_example").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the epoch example data")


def to_binary_float_labels(batch):
    """Convert the encoded binary labels to the numeric form used during training."""
    batch = dict(batch)
    batch["source"] = [float(value) for value in batch["source"]]
    return batch


binary_root = find_binary_project_root()
with (binary_root / "configs/binary_classifier_epoch_example.yaml").open() as stream:
    binary_config = yaml.safe_load(stream)["trainer"]

# Point each configured split at the repository data and apply the label conversion.
binary_data_root = binary_root / "data/gold/epoch_trainer_example"
for split in ("train", "val", "test"):
    binary_config[f"{split}_dataset_args"] = [str(binary_data_root / split)]
    binary_config[f"{split}_dataset_kwargs"]["transform"] = to_binary_float_labels

### Train the binary classifier

The architecture, optimizer, number of epochs, and input features are the same as in `epochtrainer_examples.ipynb`. This notebook writes the training artifacts to a named output directory for later loading.


In [ ]:
binary_output_path = Path("inference_example_binary_classifier")
binary_config["output_path"] = str(binary_output_path)

binary_trainer = EpochTrainer.from_config(binary_config)
binary_trainer.train()

# Use every held-out feature row when demonstrating predictions from the exports.
binary_features = torch.stack(
    [features for features, _ in binary_trainer.eval_ds]
).numpy()

### Export and load the binary classifier

Export the trained classifier in the default and PyTorch formats. Reload each export and use it to predict the held-out examples.


In [ ]:
binary_trainer.export_model("default-export")
binary_trainer.config["export_format"] = "pt"
binary_trainer.export_model("pt-export")

binary_default_model = load_default(binary_output_path / "default-export")
binary_torch_model = load_torch(binary_output_path / "pt-export")

# Both exports can now be used through the same prediction interface.
binary_default_predictions = binary_default_model.predict(binary_features)
binary_torch_predictions = binary_torch_model.predict(binary_features)
binary_default_predictions[:10], binary_torch_predictions[:10]

## Switch the task to multiclass classification

The available Cloudy data has two classes, AGN and POPSTAR. This section changes the binary configuration into a two-class multiclass configuration, then demonstrates loading and using that classifier. The same pattern extends to datasets with more encoded classes.


In [ ]:
from pathlib import Path

import torch
import yaml

from GalaxySpectrumClassifier import EpochTrainer, load_default


def find_multiclass_project_root():
    """Find the real gold-level AGN/POPSTAR dataset."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/gold/epoch_trainer_example").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the epoch example data")


multi_root = find_multiclass_project_root()
with (multi_root / "configs/binary_classifier_epoch_example.yaml").open() as stream:
    multi_config = yaml.safe_load(stream)["trainer"]

# Reuse the real split files; their source column already contains integer class labels.
multi_data_root = multi_root / "data/gold/epoch_trainer_example"
for split in ("train", "val", "test"):
    multi_config[f"{split}_dataset_args"] = [str(multi_data_root / split)]
    multi_config[f"{split}_dataset_kwargs"].pop("transform", None)

### Configure, train, and export the multiclass classifier

The settings in this cell are the changes required by the task shift: a multiclass task, a loss for class labels, and one output score per class. The remaining training settings continue to come from the epoch trainer configuration.


In [ ]:
multi_output_path = Path("inference_example_multiclass_classifier")
multi_class_count = 2
multi_config["output_path"] = str(multi_output_path)

# These are the task-specific changes from the binary configuration.
multi_config["task"] = "multiclass-classification"
multi_config["loss_type"] = "torch.nn.CrossEntropyLoss"
multi_config["model_kwargs"]["hidden_channels"] = [64, multi_class_count]
multi_config["nclasses"] = multi_class_count

multi_trainer = EpochTrainer.from_config(multi_config)
multi_trainer.train()
multi_trainer.export_model("default-export")
# Use every held-out feature row when demonstrating multiclass predictions.
multi_features = torch.stack(
    [features for features, _ in multi_trainer.eval_ds]
).numpy()

### Load the multiclass classifier and inspect predictions

Tell the loader how many encoded classes the export represents. Then inspect the class labels, predicted classes, and per-class probabilities for held-out rows.


In [ ]:
multi_model = load_default(
    multi_output_path / "default-export", nclasses=multi_class_count
)

multi_predictions = multi_model.predict(multi_features)
multi_probabilities = multi_model.predict_proba(multi_features)
# The first probability column belongs to class 0 and the second to class 1.
multi_model.classes_, multi_predictions[:10], multi_probabilities[:10]